In [ ]:

import os, psutil, platform, torch
os.environ["OMP_NUM_THREADS"]="4"; os.environ["MKL_NUM_THREADS"]="4"; os.environ["NUMEXPR_NUM_THREADS"]="4"; os.environ["TOKENIZERS_PARALLELISM"]="false"
try:
    torch.set_num_threads(4); torch.set_num_interop_threads(4)
except Exception as e:
    print("Torch thread pin warning:", e)
print("CPU count:", psutil.cpu_count(), "| RAM (GB):", round(psutil.virtual_memory().total/1e9,2))
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())



CPU count: 4 | RAM (GB): 16.78
Platform: Linux-6.1.151+-x86_64-with-glibc2.35
CUDA available: False


In [ ]:
PROJECT_ID = "instr-cs795-fall25-hqin-1"
BUCKET     = "instr-cs795-fall25-hqin-1-arasm002"
REGION = "us-central1"
EXPERIMENT = "tinyllm-phase2"

from datetime import datetime, timezone
import os
ROOT="/content/tinyllm-phase2"; MODELS=f"{ROOT}/models"; SCRIPTS=f"{ROOT}/scripts"; DATA=f"{ROOT}/data"; RESULTS=f"{ROOT}/results"; CHARTS=f"{RESULTS}/charts"
for d in (ROOT, MODELS, SCRIPTS, DATA, RESULTS, CHARTS): os.makedirs(d, exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
ARTIFACTS=f"{BUCKET}/tinyllm-phase2/{STAMP}" if BUCKET else ""
print("ROOT:", ROOT); print("BUCKET:", BUCKET or "(none)"); print("ARTIFACTS:", ARTIFACTS or "(n/a)")




ROOT: /content/tinyllm-phase2
BUCKET: instr-cs795-fall25-hqin-1-arasm002
ARTIFACTS: instr-cs795-fall25-hqin-1-arasm002/tinyllm-phase2/20251023-033039


In [ ]:
%%bash
set -e

# 1) CPU PyTorch (solid for Colab/Vertex CPU)
pip -q install --upgrade --extra-index-url https://download.pytorch.org/whl/cpu \
  "torch==2.2.2+cpu" "torchvision==0.17.2+cpu" "torchaudio==2.2.2+cpu"

# 2) Core pins that AVOID torchao and tolerate CPU torch 2.2.2
pip -q install \
  "transformers==4.45.2" \
  "accelerate==0.33.0" \
  "numpy==1.26.4" \
  "peft==0.13.2" \
  "datasets==2.21.0" \
  "evaluate==0.4.2" \
  "safetensors>=0.4.0" \
  "sentencepiece>=0.1.99" \
  "tqdm>=4.66.0" \
  "protobuf==4.25.3" \
  "pandas>=2.2.0" \
  "psutil>=5.9.0" \
  "matplotlib>=3.8.0" \
  "scipy>=1.11.0"

# 3) Quant toolkits (Py3.11 friendly & torch 2.2.2 compatible)
pip -q install "auto-gptq==0.7.1"

# AutoAWQ: prefer 0.2.8, else 0.2.5 (both lighter on torch pinning than 0.2.6)
pip -q install "autoawq==0.2.8" || pip -q install "autoawq==0.2.5"

python - << 'PY'
import sys, torch, transformers, accelerate, numpy as np, peft
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "cuda?", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("numpy:", np.__version__)
import auto_gptq; print("auto_gptq OK")
from awq import AutoAWQForCausalLM; print("autoawq/awq OK")
PY


python: 3.11.13
torch: 2.2.2+cpu cuda? False
transformers: 4.45.2
accelerate: 0.33.0
numpy: 1.26.4
auto_gptq OK
autoawq/awq OK


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 25.6.0 requires cupy-cuda12x>=12.0.0, which is not installed.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, which is not installed.
tensorflow-decision-forests 1.12.0 requires ydf>=0.11.0, which is not installed.
shap 0.48.0 requires numba>=0.54, which is not installed.
cudf-cu12 25.6.0 requires cupy-cuda12x>=12.0.0, which is not installed.
cudf-cu12 25.6.0 requires numba<0.62.0a0,>=0.59.1, which is not installed.
nx-cugraph-cu12 25.6.0 requires cupy-cuda12x>=12.0.0, which is not installed.
librosa 0.11.0 requires numba>=0.51.0, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
dask-cudf-cu12 25.6.0 requires cupy-cuda12x>=12.0.0, which is not installed.
cuml-cu

In [ ]:
import os, textwrap
ROOT="/content/tinyllm-phase2"; SCRIPTS=f"{ROOT}/scripts"; os.makedirs(SCRIPTS, exist_ok=True)

quantize_gptq = r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-gptq-4bit")
    ap.add_argument("--bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map={"": "cpu"}, trust_remote_code=True)
    calib_texts = get_calib_texts(1024)
    examples = [{"input_ids": tok(t, return_tensors="pt", truncation=True, max_length=1024)["input_ids"]} for t in calib_texts]
    qcfg = BaseQuantizeConfig(bits=args.bits, group_size=args.group_size, damp_percent=0.01, desc_act=True)
    qmodel = AutoGPTQForCausalLM.from_pretrained(base, quantize_config=qcfg)
    qmodel.quantize(examples)
    qmodel.save_pretrained(args.out); tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f: json.dump({"method":"gptq","bits":args.bits,"group_size":args.group_size}, f, indent=2)
    print("[GPTQ] Saved:", args.out)
if __name__ == "__main__":
    main()
'''

quantize_awq = r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM
def get_calib_data(tok, n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    texts = [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]
    calib_data = []
    for t in texts:
        enc = tok(t, return_tensors="pt", truncation=True, max_length=1024)
        calib_data.append({"input_ids": enc["input_ids"]})
    return calib_data
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-awq-4bit")
    ap.add_argument("--w_bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, trust_remote_code=True)
    model = AutoAWQForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map={"": "cpu"}, trust_remote_code=True)
    calib_data = get_calib_data(tok, 1024)
    model.quantize(tokenizer=tok, calib_data=calib_data, w_bits=args.w_bits, q_group_size=args.group_size, zero_point=True, version="GEMM")
    model.save_quantized(args.out, merge_lora=False, safetensors=True); tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f: json.dump({"method":"awq","bits":args.w_bits,"group_size":args.group_size}, f, indent=2)
    print("[AWQ] Saved:", args.out)
if __name__ == "__main__":
    main()
'''

evaluate_py = r'''#!/usr/bin/env python
import argparse, os, time, psutil, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline
def ppl_on_wikitext(model, tok, which="wikitext-2-raw-v1"):
    ds = load_dataset("wikitext", which, split="validation")
    text = "\\n\\n".join(ds["text"]); enc = tok(text, return_tensors="pt", truncation=False)
    input_ids = enc["input_ids"]; stride = 1024; lls = []
    model.eval()
    with torch.no_grad():
        for i in range(0, input_ids.size(1), stride):
            begin = max(i + stride - 1024, 0); end = min(i + stride, input_ids.size(1)); trg = end - i
            ids = input_ids[:, begin:end]; tgt = ids.clone(); tgt[:, :-trg] = -100
            out = model(ids, labels=tgt); lls.append(out.loss * trg)
    ppl = torch.exp(torch.stack(lls).sum() / end).item(); return ppl
def time_gen(pipe, prompt="Explain quantization in one paragraph.", gen=128):
    start = psutil.Process().memory_info().rss; t0=time.time()
    _ = pipe(prompt, max_new_tokens=gen, do_sample=False); t1=time.time()
    end = psutil.Process().memory_info().rss; tot=t1-t0
    return {"latency_ms_per_token": (tot/gen)*1000.0, "throughput_tok_s": gen/tot if tot>0 else float("nan"), "peak_ram_gb": max(start,end)/(1024**3)}
def load_model(which, model_name=None, model_dir=None):
    if which=="fp16":
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map={"": "cpu"}, trust_remote_code=True); return model, tok
    tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype="auto", device_map={"": "cpu"}, trust_remote_code=True); return model, tok
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--which", required=True, choices=["fp16","gptq","awq"])
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct"); ap.add_argument("--model_dir", default=None)
    ap.add_argument("--wiki", default="wikitext-2-raw-v1", choices=["wikitext-2-raw-v1","wikitext-103-raw-v1"])
    args = ap.parse_args()
    model, tok = load_model(args.which, model_name=args.model, model_dir=args.model_dir)
    ppl = ppl_on_wikitext(model, tok, args.wiki)
    pipe = TextGenerationPipeline(model=model, tokenizer=tok, device=-1)
    perf = time_gen(pipe)
    row = {"method": args.which, "model_dir_or_name": args.model if args.which=="fp16" else args.model_dir, "ppl": ppl, **perf}
    import pandas as pd, os
    os.makedirs("results", exist_ok=True); csv="results/metrics.csv"
    df=pd.DataFrame([row]);
    if os.path.exists(csv): df=pd.concat([pd.read_csv(csv), df], ignore_index=True)
    df.to_csv(csv, index=False); print("[Eval] wrote", csv)
if __name__=="__main__": main()
'''

export_py = r'''#!/usr/bin/env python
# Prints suggested GGUF convert command for llama.cpp
import argparse
ap=argparse.ArgumentParser()
ap.add_argument("--hf_model", default="microsoft/Phi-3-mini-4k-instruct")
ap.add_argument("--out_gguf", default="/content/tinyllm-phase2/models/phi3mini-f16.gguf")
ap.add_argument("--llama_dir", default="/content/llama.cpp")
args=ap.parse_args()
print("Run these commands in a shell:")
print(f"python {args.llama_dir}/convert-hf-to-gguf.py --model {args.hf_model} --outfile {args.out_gguf} --outtype f16")
print(f"./quantize {args.out_gguf} /content/tinyllm-phase2/models/phi3mini-q4_0.gguf q4_0")
print("./main -m /content/tinyllm-phase2/models/phi3mini-q4_0.gguf -p 'Explain quantization in 2 sentences.' -n 128 --threads 4")
'''

open(f"{SCRIPTS}/quantize_gptq.py","w").write(quantize_gptq)
open(f"{SCRIPTS}/quantize_awq.py","w").write(quantize_awq)
open(f"{SCRIPTS}/evaluate.py","w").write(evaluate_py)
open(f"{SCRIPTS}/export_to_gguf.py","w").write(export_py)
print("Scripts written to", SCRIPTS)


Scripts written to /content/tinyllm-phase2/scripts


In [ ]:
%%bash
set -e
cat > /content/tinyllm-phase2/scripts/quantize_gptq.py << 'PY'
#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:4096]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-gptq-4bit")
    ap.add_argument("--bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, trust_remote_code=True)
    calib_texts = get_calib_texts(1024)
    examples = [{"input_ids": tok(t, return_tensors="pt", truncation=True, max_length=1024)["input_ids"]}
                for t in calib_texts]

    qcfg = BaseQuantizeConfig(bits=args.bits, group_size=args.group_size,
                              damp_percent=0.01, desc_act=True)

    # IMPORTANT: pass model ID (string), not a model object
    qmodel = AutoGPTQForCausalLM.from_pretrained(
        args.model,
        quantize_config=qcfg,
        trust_remote_code=True,
        device_map={"": "cpu"},
    )
    qmodel.quantize(examples)
    qmodel.save_pretrained(args.out)
    tok.save_pretrained(args.out)

    with open(os.path.join(args.out, "quant_info.json"), "w") as f:
        json.dump({"method": "gptq", "bits": args.bits, "group_size": args.group_size}, f, indent=2)
    print("[GPTQ] Saved:", args.out)

if __name__ == "__main__":
    main()
PY
chmod +x /content/tinyllm-phase2/scripts/quantize_gptq.py


In [6]:
%cd /content/tinyllm-phase2
!python scripts/quantize_gptq.py --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-gptq-4bit --bits 4 --group_size 128
!python scripts/quantize_awq.py  --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-awq-4bit  --w_bits 4 --group_size 128
!python scripts/evaluate.py --which fp16 --model microsoft/Phi-3-mini-4k-instruct --wiki wikitext-2-raw-v1
!python scripts/evaluate.py --which gptq --model_dir models/phi3mini-gptq-4bit --wiki wikitext-2-raw-v1
!python scripts/evaluate.py --which awq  --model_dir models/phi3mini-awq-4bit  --wiki wikitext-2-raw-v1

/content/tinyllm-phase2
CUDA extension not installed.
CUDA extension not installed.
Traceback (most recent call last):
  File "/content/tinyllm-phase2/scripts/quantize_gptq.py", line 45, in <module>
    main()
  File "/content/tinyllm-phase2/scripts/quantize_gptq.py", line 30, in main
    qmodel = AutoGPTQForCausalLM.from_pretrained(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/auto_gptq/modeling/auto.py", line 75, in from_pretrained
    model_type = check_and_get_model_type(pretrained_model_name_or_path, trust_remote_code)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/auto_gptq/modeling/_utils.py", line 305, in check_and_get_model_type
    raise TypeError(f"{config.model_type} isn't supported yet.")
TypeError: phi3 isn't supported yet.
Fetching 19 files: 100% 19/19 [00:00<00:00, 18182.02it/s]
`flash-attention` package not found, consider in